# Fourier-feature CFM for an eroded mountain landscape

Fit the hydraulic-erosion heightmap saved in `C:/Users/shich/Src/smooth_chaos/noise.ipynb` using **one hidden ReLU layer** conditioned on Fourier coordinate features. The target is the saved `eroded_height` surface. Its input field is reconstructed as `eroded_height - height_diff` from the two outputs of the same source cell. Both are bundled in `_shared/data/noise_hydraulic_terrain.npz`; there is no external runtime dependency. `TerrainConfig(source="reference", resolution=128)` loads these heightmaps directly. The `Saw`, `MountainGen`, and `hydraulic_erosion` definitions are also available in `_shared/mountains.py` for `source="generated"` targets.

The coordinate embedding contains normalized raw `(x, y)` plus sine/cosine bands at frequencies **1, 2, 4, 8, 16, 32** along four directions. Only coordinates receive Fourier features. The noisy height is concatenated to this embedding before `Linear -> ReLU -> Linear`; Fourier projections are fixed buffers, not trainable parameters.

**Training:** standardize the eroded height field, sample mini-batches of grid coordinates, draw `t ~ Uniform(0, 1)` and `z ~ Normal(0, 1)`, construct `s_t=(1-t)z+t*h_normalized`, and minimize `MSE(residual_hat, h_normalized-s_t)`. Time is not a network input. Coordinate features are cached because their frequencies are fixed.

**Inference:** use **64 Euler steps** with velocity `residual_hat/(1-t)`, starting from a zero normalized-height state at every coordinate. Convert predictions back to the original height units. The sampler never receives target heights. Coordinate features are computed once per inference chunk and reused across Euler steps.

The live renderer evaluates a 96x96 preview and shows height MSE, target and prediction as heatmaps with the same height color scale, and a height-error map. After training, evaluate the entire terrain grid and compare both 3D surfaces with shared height/color scales. Grid MSE measures reconstruction of the heightmap, not a held-out test set. Weights, fixed Fourier buffers, target terrain, normalization, and configuration are saved together for repeatable inference.

In [1]:
from pathlib import Path
import sys

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import numpy as np
import torch
from IPython.display import display, Markdown
from _shared.mountains import TerrainConfig, generate_terrain
from _shared.fourier_mountain import (
    MountainFitConfig,
    MountainExperiment,
    surface_comparison,
    load_mountain_checkpoint,
)
from _shared.conditional_flow_matching import FlowMatchingConfig
from _shared.visualization import train_live

torch.set_num_threads(1)

In [2]:
terrain_config = TerrainConfig(
    source="reference",
    resolution=128,
)
fit_config = MountainFitConfig(
    hidden_size=512,
    frequencies=(1, 2, 4, 8, 16, 32),
    directions=4,
    total_steps=32_000,
    batch_size=1024,
    learning_rate=0.001,
    seed=42,
    preview_resolution=96,
    inference_batch_size=2048,
    device="cpu",  # Set to "cuda" when using a CUDA-enabled PyTorch installation.
)
flow_config = FlowMatchingConfig(
    train_sampling_steps=None,
    eval_sampling_steps=1,
)
target_fps = 0.5
output_dir = project_root / "notebooks" / "outputs"
checkpoint_path = output_dir / "fourier_mountain_cfm.pt"

In [3]:
terrain = generate_terrain(terrain_config)
figure = surface_comparison(terrain)
figure.show(config={"responsive": True, "displayModeBar": False})

In [4]:
mountain_experiment = MountainExperiment(terrain, fit_config, flow_config)
parameter_count = sum(
    parameter.numel() for parameter in mountain_experiment.model.parameters()
)
display(
    Markdown(
        f"**Trainable parameters:** {parameter_count:,} | "
        f"**Coordinate features:** {mountain_experiment.model.features.output_size} | "
        f"**Training grid:** {terrain_config.resolution} x {terrain_config.resolution}"
    )
)
try:
    await train_live(mountain_experiment, target_fps=target_fps)
finally:
    # Save the current model and terrain even when training is interrupted.
    mountain_experiment.save(checkpoint_path)

**Trainable parameters:** 27,137 | **Coordinate features:** 50 | **Training grid:** 128 x 128

## Load saved weights for inference

In a fresh kernel, run imports and configuration, then this cell instead of terrain generation and training. The checkpoint includes the exact terrain and normalization. Run the full-grid inference cell afterward.

In [5]:
# Optional: restore the checkpoint without training or regenerating the terrain.
mountain_experiment = load_mountain_checkpoint(
    checkpoint_path, device=fit_config.device
)
terrain = mountain_experiment.terrain
from _shared.pull_render import show_current

await show_current(mountain_experiment, target_fps=target_fps)

## Standard Fourier MLP baseline

Train a standard MSE regressor on the **same terrain and Fourier features** as the CFM model. The baseline uses the same hidden width, seed, Adam learning rate, mini-batch indices, height normalization, and optimizer-update budget. It predicts normalized height directly, with one forward pass for inference. CFM additionally receives noisy height, so its input layer has `hidden_size` more parameters.

Run the cells below after training or loading the CFM model. If no CFM experiment is in memory, its checkpoint is loaded automatically. The baseline is saved separately as `fourier_mountain_standard.pt`; CFM weights and cells are preserved. The comparison reports full-grid MSE and MAE in original height units, with actual completed training steps and parameter counts, then shows target, both fits, and both signed-error maps on shared color scales.

In [6]:
import importlib
import json
from dataclasses import asdict
import _shared.fourier_mountain as mountain_models

# Load the model and visualization definitions used by these cells.
mountain_models = importlib.reload(mountain_models)
if "mountain_experiment" not in globals():
    mountain_experiment = mountain_models.load_mountain_checkpoint(
        checkpoint_path,
        device=fit_config.device,
    )
comparison_terrain = mountain_experiment.terrain
standard_config = mountain_experiment.config
standard_checkpoint_path = output_dir / "fourier_mountain_standard.pt"

In [8]:
standard_experiment = mountain_models.StandardMountainExperiment(
    comparison_terrain, standard_config
)
standard_parameters = sum(
    parameter.numel() for parameter in standard_experiment.model.parameters()
)
display(
    Markdown(
        f"**Standard MLP parameters:** {standard_parameters:,} | "
        f"**Hidden neurons:** {standard_config.hidden_size} | "
        f"**Training steps:** {standard_config.total_steps:,}"
    )
)
try:
    await train_live(standard_experiment, target_fps=target_fps)
finally:
    standard_experiment.save(standard_checkpoint_path)

**Standard MLP parameters:** 26,625 | **Hidden neurons:** 512 | **Training steps:** 32,000

In [9]:
cfm_prediction = mountain_experiment.predict_full_grid()
standard_prediction = standard_experiment.predict_full_grid()
target = comparison_terrain.eroded_height
comparison_metrics = []
for method, experiment, prediction, inference_steps in (
    (
        "CFM",
        mountain_experiment,
        cfm_prediction,
        mountain_experiment.flow_config.eval_sampling_steps,
    ),
    ("Standard MLP", standard_experiment, standard_prediction, 0),
):
    error = prediction - target
    comparison_metrics.append(
        {
            "method": method,
            "hidden_size": experiment.config.hidden_size,
            "parameters": sum(
                parameter.numel() for parameter in experiment.model.parameters()
            ),
            "training_steps": experiment.progress.step,
            "eval_sampling_steps": inference_steps,
            "grid_mse": float(np.mean(error**2)),
            "grid_mae": float(np.mean(np.abs(error))),
        }
    )
table = [
    "| Method | Hidden neurons | Parameters | Training steps | Inference | Grid MSE | Grid MAE |",
    "|:--|--:|--:|--:|:--|--:|--:|",
]
for row in comparison_metrics:
    inference = (
        f"{row['eval_sampling_steps']} Euler steps"
        if row["method"] == "CFM"
        else "Direct"
    )
    table.append(
        f"| {row['method']} | {row['hidden_size']} | {row['parameters']:,} | {row['training_steps']:,} | "
        f"{inference} | {row['grid_mse']:.8g} | {row['grid_mae']:.8g} |"
    )
display(Markdown("### Full-grid comparison\n\n" + "\n".join(table)))
figure = mountain_models.comparison_heatmaps(
    comparison_terrain,
    cfm_prediction,
    standard_prediction,
    mountain_experiment.flow_config.eval_sampling_steps,
)
figure.show(config={"responsive": True, "displayModeBar": False})
output_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    output_dir / "fourier_mountain_comparison.npz",
    x=comparison_terrain.x,
    y=comparison_terrain.y,
    target=target,
    cfm_prediction=cfm_prediction,
    standard_prediction=standard_prediction,
    cfm_error=cfm_prediction - target,
    standard_error=standard_prediction - target,
)
(output_dir / "fourier_mountain_comparison_metrics.json").write_text(
    json.dumps(
        {"fit_config": asdict(standard_config), "metrics": comparison_metrics}, indent=2
    )
    + "\n",
    encoding="utf-8",
)

### Full-grid comparison

| Method | Hidden neurons | Parameters | Training steps | Inference | Grid MSE | Grid MAE |
|:--|--:|--:|--:|:--|--:|--:|
| CFM | 512 | 27,137 | 32,000 | 1 Euler steps | 2.7934971e-06 | 0.0014005203 |
| Standard MLP | 512 | 26,625 | 32,000 | Direct | 3.2641012e-07 | 0.00045441647 |

826

### Load baseline weights for comparison

Run this cell instead of baseline training to restore saved baseline weights. Then run the full-grid comparison cell above.

In [ ]:
standard_experiment = mountain_models.load_mountain_checkpoint(
    standard_checkpoint_path,
    device=standard_config.device,
)